# Experiment: xNES vs CMA-ES Workbench

This notebook compares the current `leitwerk.xnes.XNES` implementation against CMA-ES on a noisy Ackley stress test.

- shared population size: xNES default heuristic via `_default_sample_count(None, d)`
- both optimizers start from the same mean and isotropic scale
- xNES uses the current built-in `XNES.tell` defaults from this branch
- the final plot shows best-so-far fitness vs generation with a linear x-axis and log-scaled y-axis

The goal is a direct readout of current xNES behavior against CMA-ES under the same stochastic objective.


In [ ]:
from __future__ import annotations

from collections import Counter
from collections.abc import Callable

import cma
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from leitwerk import XNES, XNESStatus
from leitwerk.xnes import _default_sample_count


In [ ]:
# Main knobs.
DIMENSION = 100
MAX_ITERATIONS = 1000
NUM_TRIALS = 10
MASTER_SEED = 1234

# Problem setup. This is not optimizer tuning; both methods see the same initial search state.
INITIAL_SIGMA = 2.5
START_VALUE = 6.0
LOG_EPS = 1e-16
NOISE_SCALE = 1e-2

POPULATION_SIZE = _default_sample_count(None, DIMENSION)
X0 = np.full(DIMENSION, START_VALUE, dtype=float)
TRIAL_SEEDS = MASTER_SEED + np.arange(NUM_TRIALS, dtype=int)

print(
    {
        'dimension': DIMENSION,
        'population_size': POPULATION_SIZE,
        'max_iterations': MAX_ITERATIONS,
        'num_trials': NUM_TRIALS,
        'initial_sigma': INITIAL_SIGMA,
        'noise_scale': NOISE_SCALE,
    }
)


In [ ]:
def ackley(x: np.ndarray, a: float = 20.0, b: float = 0.2, c: float = 2.0 * np.pi) -> float:
    x = np.asarray(x, dtype=float)
    mean_sq = float(np.mean(x**2))
    mean_cos = float(np.mean(np.cos(c * x)))
    return float(-a * np.exp(-b * np.sqrt(mean_sq)) - np.exp(mean_cos) + a + np.e)


def make_noisy_ackley(seed: int, noise_scale: float = NOISE_SCALE) -> Callable[[np.ndarray], float]:
    rng = np.random.default_rng(seed)

    def objective(x: np.ndarray) -> float:
        base = ackley(x)
        noise = noise_scale * (1.0 + base) * abs(float(rng.standard_normal()))
        return float(base + noise)

    return objective


OBJECTIVE_FACTORY: Callable[[int], Callable[[np.ndarray], float]] = make_noisy_ackley


In [ ]:
def run_xnes_trial(
    objective: Callable[[np.ndarray], float],
    x0: np.ndarray,
    sigma0: float,
    population_size: int,
    max_iterations: int,
    seed: int,
) -> tuple[np.ndarray, str]:
    xnes = XNES(x0, sigma0)
    rng = np.random.default_rng(seed)

    history = np.empty(max_iterations, dtype=float)
    best_so_far = np.inf
    final_status = XNESStatus.OK.name

    for iteration in range(max_iterations):
        samples = xnes.ask(population_size, rng)
        candidates = xnes.transform(samples).T
        values = np.array([float(objective(candidate)) for candidate in candidates], dtype=float)
        status = xnes.tell(samples, np.argsort(values).tolist())
        best_so_far = min(best_so_far, float(np.min(values)))
        history[iteration] = best_so_far
        if status is not XNESStatus.OK:
            final_status = status.name
            history[iteration:] = best_so_far
            break
    else:
        return history, final_status

    return history, final_status


def run_cma_trial(
    objective: Callable[[np.ndarray], float],
    x0: np.ndarray,
    sigma0: float,
    population_size: int,
    max_iterations: int,
    seed: int,
) -> tuple[np.ndarray, str]:
    es = cma.CMAEvolutionStrategy(
        x0,
        sigma0,
        {
            'popsize': population_size,
            'seed': int(seed),
            'verb_disp': 0,
            'verb_log': 0,
        },
    )

    history = np.empty(max_iterations, dtype=float)
    best_so_far = np.inf
    final_status = 'MAX_ITERATIONS'

    for iteration in range(max_iterations):
        stop_reasons = es.stop()
        if stop_reasons:
            final_status = ', '.join(sorted(str(reason) for reason in stop_reasons))
            if not np.isfinite(best_so_far):
                best_so_far = float(objective(x0))
            history[iteration:] = best_so_far
            break

        candidates = es.ask()
        values = [float(objective(np.asarray(candidate, dtype=float))) for candidate in candidates]
        es.tell(candidates, values)
        best_so_far = min(best_so_far, min(values))
        history[iteration] = best_so_far
    else:
        return history, final_status

    return history, final_status


def run_suite(
    objective_factory: Callable[[int], Callable[[np.ndarray], float]],
    x0: np.ndarray,
    sigma0: float,
    population_size: int,
    max_iterations: int,
    trial_seeds: np.ndarray,
) -> tuple[dict[str, np.ndarray], dict[str, list[str]]]:
    optimizer_specs = [
        ('xNES', lambda seed: run_xnes_trial(objective_factory(seed), x0, sigma0, population_size, max_iterations, seed)),
        ('CMA-ES', lambda seed: run_cma_trial(objective_factory(seed), x0, sigma0, population_size, max_iterations, seed)),
    ]

    curves = {}
    statuses = {}
    for label, runner in optimizer_specs:
        runs = [runner(int(seed)) for seed in tqdm(trial_seeds, desc=label, leave=False)]
        curves[label] = np.vstack([history for history, _ in runs])
        statuses[label] = [status for _, status in runs]
    return curves, statuses


In [ ]:
curves, statuses = run_suite(
    objective_factory=OBJECTIVE_FACTORY,
    x0=X0,
    sigma0=INITIAL_SIGMA,
    population_size=POPULATION_SIZE,
    max_iterations=MAX_ITERATIONS,
    trial_seeds=TRIAL_SEEDS,
)

for label, values in curves.items():
    print(label, values.shape, dict(Counter(statuses[label])))


In [ ]:
summary = {
    label: {
        'median_final_fitness': float(np.median(values[:, -1])),
        'best_final_fitness': float(np.min(values[:, -1])),
        'worst_final_fitness': float(np.max(values[:, -1])),
        'statuses': dict(Counter(statuses[label])),
    }
    for label, values in curves.items()
}
summary


In [ ]:
iterations = np.arange(1, MAX_ITERATIONS + 1, dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))
palette = {'xNES': '#d62728', 'CMA-ES': '#1f77b4'}

for label, values in curves.items():
    clipped = np.maximum(values, LOG_EPS)
    for run_index, run in enumerate(clipped):
        ax.plot(
            iterations,
            run,
            color=palette[label],
            alpha=0.5,
            linewidth=1.5,
            label=label if run_index == 0 else None,
        )

ax.set_xscale('linear')
ax.set_yscale('log')
ax.set_xlabel('generation')
ax.set_ylabel('best-so-far fitness')
ax.set_title(f'Noisy Ackley comparison, d={DIMENSION}, popsize={POPULATION_SIZE}, trials={NUM_TRIALS}')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
